In [3]:
import math

# Normal distribution
def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def normal_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

# Black-Scholes Call
def black_scholes_call(S, K, r, q, sigma, T):
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    call_price = S * math.exp(-q * T) * normal_cdf(d1) \
               - K * math.exp(-r * T) * normal_cdf(d2)
    return call_price

# Vega 
def call_vega(S, K, r, q, sigma, T):
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    return S * math.exp(-q * T) * normal_pdf(d1) * sqrtT

#Newton's Method 
def implied_volatility(
    market_price,
    S, K, r, q, T,
    initial_guess=0.5,
    tol_price=1e-12,
    tol_sigma=1e-12,
    max_iterations=100
):
    sigma_new = initial_guess
    sigma_old = initial_guess - 1.0  
    iteration = 0

    while (abs(sigma_new - sigma_old) > tol_sigma or
           abs(black_scholes_call(S, K, r, q, sigma_new, T) - market_price) > tol_price):

        iteration += 1
        if iteration > max_iterations:
            raise Exception("Newtons method did not converge")

        sigma_old = sigma_new

        price = black_scholes_call(S, K, r, q, sigma_old, T)
        vega = call_vega(S, K, r, q, sigma_old, T)

        sigma_new = sigma_old - (price - market_price) / vega

        if sigma_new <= 0:
            sigma_new = 1e-8

    return sigma_new, iteration


S0 = 40.0
K = 40.0
T = 5.0/12.0
r = 0.025
q = 0.01
market_price = 2.75


sigma_star, iterations = implied_volatility(
    market_price, S0, K, r, q, T, initial_guess=0.5
)

print("Implied volatility:", round(sigma_star, 6))
print("Iterations:", iterations)
## Implied volatility: 0.280756
## Iterations: 4

Implied volatility: 0.256903
Iterations: 4
